In [1]:
import re
from dataclasses import dataclass
from typing import Optional, Literal

ResolutionDataType = Literal[
    "candle_ohlcv",
    "daily_metric",
    "ranking_snapshot",
    "aggregate_total",
    "equity_event",
    "other"
]

IntervalSource = Literal[
    "explicit",
    "default_1m",
    "default_1d",
    "none"
]

@dataclass(frozen=True)
class ResolutionRouting:
    data_type: ResolutionDataType
    interval: Optional[str]          # "1m", "5m", "1d", etc.
    interval_source: IntervalSource  # explicit vs default
    notes: Optional[str] = None


# --- helpers ---

_INTERVAL_PATTERNS = [
    ("1m", re.compile(r"\b1\s*m(in(ute)?)?\b|\b1m\b|one[-\s]?minute", re.IGNORECASE)),
    ("5m", re.compile(r"\b5\s*m(in(ute)?)?\b|\b5m\b|five[-\s]?minute", re.IGNORECASE)),
    ("15m", re.compile(r"\b15\s*m(in(ute)?)?\b|\b15m\b|fifteen[-\s]?minute", re.IGNORECASE)),
    ("30m", re.compile(r"\b30\s*m(in(ute)?)?\b|\b30m\b|thirty[-\s]?minute", re.IGNORECASE)),
    ("1h", re.compile(r"\b1\s*h(our)?\b|\b1h\b|one[-\s]?hour", re.IGNORECASE)),
    ("4h", re.compile(r"\b4\s*h(our)?\b|\b4h\b|four[-\s]?hour", re.IGNORECASE)),
    ("1d", re.compile(r"\b1\s*d(ay)?\b|\b1d\b|daily\b", re.IGNORECASE)),
]

# Signals that this is an OHLC/candle market even if interval isn't stated
_CANDLE_SIGNALS = re.compile(
    r"\b(candle|candles|ohlc|open|high|low|close|closing price)\b",
    re.IGNORECASE
)

# Daily metric signals (your two USDT cases + FDV snapshots etc.)
_DAILY_METRIC_SIGNALS = re.compile(
    r"\b(daily|historical data|market cap|marketcap|fdv)\b",
    re.IGNORECASE
)

# Ranking snapshot signals (top 100 etc.)
_RANKING_SIGNALS = re.compile(
    r"\b(top\s*100|rank|ranking)\b",
    re.IGNORECASE
)

# Aggregate total signals (amount raised, total raised)
_AGG_TOTAL_SIGNALS = re.compile(
    r"\b(total amount raised|amount raised|total raised|funds raised)\b",
    re.IGNORECASE
)

# Equity/IPO event signals
_EQUITY_EVENT_SIGNALS = re.compile(
    r"\b(ipo|first trading day|outstanding shares|primary exchange)\b",
    re.IGNORECASE
)


def _extract_explicit_interval(terms: str) -> Optional[str]:
    if not terms:
        return None
    for code, pat in _INTERVAL_PATTERNS:
        if pat.search(terms):
            # Note: "daily" will map to 1d; keep that for daily metrics
            return code
    return None


def route_resolution_terms(source: Optional[str], terms: Optional[str]) -> ResolutionRouting:
    """
    Deterministic routing:
    1) If OHLC/candle is referenced -> candle_ohlcv, interval explicit or default 1m
    2) Else if equity/ipo -> equity_event (no interval)
    3) Else if aggregate totals -> aggregate_total (no interval)
    4) Else if rankings -> ranking_snapshot (no interval)
    5) Else if daily metric -> daily_metric (interval 1d)
    6) Else other
    """
    s = (source or "")
    t = (terms or "")
    blob = (s + "\n" + t).lower()

    explicit_interval = _extract_explicit_interval(blob)

    # 1) Candle markets (the main case)
    if _CANDLE_SIGNALS.search(blob):
        if explicit_interval and explicit_interval != "1d":
            return ResolutionRouting(
                data_type="candle_ohlcv",
                interval=explicit_interval,
                interval_source="explicit",
                notes=None
            )
        # default for candles
        return ResolutionRouting(
            data_type="candle_ohlcv",
            interval="1m",
            interval_source="default_1m",
            notes="no explicit interval; defaulted to 1m"
        )

    # 2) Equity/IPO events (not candles)
    if _EQUITY_EVENT_SIGNALS.search(blob):
        return ResolutionRouting(
            data_type="equity_event",
            interval=None,
            interval_source="none",
            notes="equity/IPO style resolution"
        )

    # 3) Aggregate totals (not candles)
    if _AGG_TOTAL_SIGNALS.search(blob):
        return ResolutionRouting(
            data_type="aggregate_total",
            interval=None,
            interval_source="none",
            notes="aggregate total (funding/raised) resolution"
        )

    # 4) Rankings (not candles)
    if _RANKING_SIGNALS.search(blob):
        return ResolutionRouting(
            data_type="ranking_snapshot",
            interval=None,
            interval_source="none",
            notes="ranking snapshot resolution"
        )

    # 5) Daily metric (CoinGecko historical data etc.)
    # If explicit says 1d OR daily signals
    if (explicit_interval == "1d") or _DAILY_METRIC_SIGNALS.search(blob):
        return ResolutionRouting(
            data_type="daily_metric",
            interval="1d",
            interval_source="default_1d" if explicit_interval is None else "explicit",
            notes="daily metric / historical data resolution"
        )

    return ResolutionRouting(
        data_type="other",
        interval=None,
        interval_source="none",
        notes="no clear candle/daily/ranking/aggregate/equity signals"
    )


In [2]:
import json
import re
from dataclasses import dataclass
from typing import Optional, Iterable, Self
from pathlib import Path

@dataclass(frozen=True)
class CoinUniverse:
    symbols: set[str]
    name_to_symbol: dict[str, str]

    @classmethod
    def from_json(cls, path: str = "coins_universe.json") -> Self:
        with open(path, "r", encoding="utf-8") as f:
            xs = json.load(f)
        symbols = set()
        name_to_symbol = {}
        for x in xs:
            sym = (x.get("symbol") or "").strip().lower()
            name = (x.get("name") or "").strip().lower()
            if sym:
                symbols.add(sym)
            if name and sym:
                name_to_symbol[name] = sym
        return cls(symbols=symbols, name_to_symbol=name_to_symbol)




@dataclass(frozen=True)
class ProjectUniverse:
    """
    Deterministic mapping of known project identifiers -> canonical labels.

    Example JSON input entries:
      {"key": "metamask", "label": "MetaMask", "type": "project"}
      {"key": "usd.ai", "label": "USD.AI", "type": "project"}

    Notes:
    - We only use "key" and "label". Other fields are ignored (but allowed).
    - Matching is exact on normalized tokens/phrases (no fuzzy matching).
    """
    key_to_label: dict[str, str]

    @classmethod
    def from_json(cls, path: str | Path="projects_universe.json") -> Self:
        """
        Load a ProjectUniverse from a JSON file containing a list of objects.

        Each object should contain:
          - key: str (normalized matching key; we normalize again defensively)
          - label: str (canonical label to output)
        """
        path = Path(path)
        with path.open("r", encoding="utf-8") as f:
            xs = json.load(f)

        if not isinstance(xs, list):
            raise ValueError(f"{path} must contain a JSON list of objects")

        key_to_label: dict[str, str] = {}
        for i, x in enumerate(xs):
            if not isinstance(x, dict):
                raise ValueError(f"{path}[{i}] must be an object, got {type(x).__name__}")

            raw_key = x.get("key")
            raw_label = x.get("label")

            if not isinstance(raw_key, str) or not raw_key.strip():
                # skip silently: allows you to keep comments/partials if you want
                continue
            if not isinstance(raw_label, str) or not raw_label.strip():
                continue

            key = normalize_text(raw_key)
            label = raw_label.strip()

            if not key:
                continue

            key_to_label[key] = label

        return cls(key_to_label=key_to_label)

    def match(self, text: str, max_phrase_len: int = 4) -> str | None:
        """
        Attempt to match `text` against known project keys.

        Strategy (deterministic):
        1) Exact token match
        2) Exact phrase match for 2..max_phrase_len tokens (longest-first)

        Returns:
          canonical label if matched, else None.
        """
        q = normalize_text(text)
        if not q:
            return None

        toks = q.split()

        # 1) exact token match
        for t in toks:
            label = self.key_to_label.get(t)
            if label is not None:
                return label

        # 2) exact phrase match (longest-first)
        L = min(max_phrase_len, len(toks))
        for n in range(L, 1, -1):
            for i in range(len(toks) - n + 1):
                phrase = " ".join(toks[i : i + n])
                label = self.key_to_label.get(phrase)
                if label is not None:
                    return label

        return None

In [3]:
def normalize_text(s: str) -> str:
    s = (s or "").lower()
    # keep $ and digits, strip most punctuation to spaces
    s = re.sub(r"[^a-z0-9\$\s\-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def words(s: str) -> list[str]:
    return normalize_text(s).split()

def extract_candidates(question: str) -> list[str]:
    q = normalize_text(question)
    cands: list[str] = []

    # 1) after "will" (grab next 1–4 tokens)
    m = re.search(r"\bwill\s+([a-z0-9\-]+(?:\s+[a-z0-9\-]+){0,3})", q)
    if m:
        cands.append(m.group(1))

    # 2) before "fdv"
    m = re.search(r"\b([a-z0-9\-]+(?:\s+[a-z0-9\-]+){0,3})\s+fdv\b", q)
    if m:
        cands.append(m.group(1))

    # 3) before "up or down"
    m = re.search(r"\b([a-z0-9\-]+(?:\s+[a-z0-9\-]+){0,3})\s+up\s+or\s+down\b", q)
    if m:
        cands.append(m.group(1))

    # extra: "X market cap" / "X price" / "X dominance"
    for kw in ["market cap", "price", "dominance", "tvl", "volume", "supply"]:
        m = re.search(rf"\b([a-z0-9\-]+(?:\s+[a-z0-9\-]+){{0,2}})\s+{kw}\b", q)
        if m:
            cands.append(m.group(1))

    # extra: direct all-caps tickers often appear as tokens in the original text;
    # we’ll also return the whole question for token scanning (handled later)
    cands.append(q)
    return [c.strip() for c in cands if c and c.strip()]

def match_symbol_from_candidate(cand: str, U: CoinUniverse) -> Optional[str]:
    toks = words(cand)

    # exact symbol token match
    for t in toks:
        if t in U.symbols:
            return t.upper()

    # exact name match (single or multiword)
    # check longest spans first
    for n in range(min(4, len(toks)), 0, -1):
        for i in range(len(toks) - n + 1):
            phrase = " ".join(toks[i:i+n])
            if phrase in U.name_to_symbol:
                return U.name_to_symbol[phrase].upper()

    return None

def parse_underlying_symbol(question: str, CU: CoinUniverse, PU: ProjectUniverse) -> Optional[str]:
    for cand in extract_candidates(question):
        sym = match_symbol_from_candidate(cand, CU)
        if sym:
            return sym
        project = PU.match(question)
        if project:
            return project
    return None

def parse_metric(question: str) -> str:
    q = normalize_text(question)
    if "fdv" in q:
        return "fdv"
    if "market cap" in q or "marketcap" in q:
        return "market_cap"
    if "dominance" in q:
        return "dominance"
    if "tvl" in q:
        return "tvl"
    if "up or down" in q:
        return "direction"
    if "price" in q or "$" in q:
        return "price"
    return "unknown"

In [4]:
coin_universe = CoinUniverse.from_json()
project_universe = ProjectUniverse.from_json()

FileNotFoundError: [Errno 2] No such file or directory: 'coins_universe.json'

In [ ]:
import re
import json
import pandas as pd
from datetime import datetime, timezone
from urllib.parse import urlparse

# from src.polymarket.clients import GammaClient

gamma = GammaClient()

def get_crypto_tag_id() -> int:
    tag = gamma.get_tag_by_slug("crypto")  # robust
    return int(tag["id"])

def safe_dt(x):
    if not x:
        return None
    try:
        return pd.to_datetime(x, utc=True)
    except Exception:
        return None

def normalize_outcomes(outcomes):
    if outcomes is None:
        return None
    if isinstance(outcomes, str):
        try:
            outcomes = json.loads(outcomes)
        except Exception:
            return None
    if isinstance(outcomes, list):
        return [str(o) for o in outcomes]
    return None

def classify_edge_or_range(question: str, outcomes: list[str] | None) -> str:
    q = (question or "").lower()

    # Strong signal: yes/no outcomes
    if outcomes and set(o.strip().lower() for o in outcomes) == {"yes", "no"}:
        # Still need to decide if edge/range; yes/no usually implies edge (but could be event)
        # We'll classify as "edge" if question mentions a numeric threshold or above/below language.
        if re.search(r"\$\s*\d|\babove\b|\bbelow\b|\bover\b|\bunder\b|\bgreater\b|\bless\b", q):
            return "edge"
        return "unknown"

    # Range signals: multiple bins, inequality symbols, or "between"
    if outcomes and len(outcomes) >= 3:
        # if outcomes include ranges or inequality symbols, likely range
        joined = " ".join(outcomes).lower()
        if any(tok in joined for tok in ["-", "–", "to", "between", "<", ">", "≤", "≥"]):
            return "range"

    # Text-only heuristic
    if any(w in q for w in [" between ", " range ", " in a range "]):
        return "range"
    if any(w in q for w in [" above ", " below ", " over ", " under ", " greater than ", " less than "]):
        return "edge"

    return "unknown"

def extract_resolution_source_and_terms(mkt: dict) -> tuple[str | None, str | None]:
    # Gamma fields vary; we’ll try a few
    source = mkt.get("resolutionSource") or mkt.get("resolution_source") or mkt.get("oracle") or None

    # Terms/rules: prefer explicit rules field, else description
    terms = mkt.get("rules") or mkt.get("resolution") or mkt.get("description") or None

    # If source missing, try to infer from terms (common: Chainlink, Coinbase, Binance, Coingecko)
    blob = ((source or "") + "\n" + (terms or "")).lower()
    if source is None:
        for cand in ["chainlink", "coinbase", "binance", "coingecko", "kraken", "bitstamp", "okx", "bybit"]:
            if cand in blob:
                source = cand

    if source is None and terms:
        # simple, robust URL extraction
        m = re.search(r"https?://[^\s)>\]]+", terms)
        if m:
            url = m.group(0)
            source = url

    # Trim terms to keep outputs manageable
    if terms:
        terms = terms.strip()


    if source:
        source = str(source).strip()

    return source, terms

def extract_resolution_date(mkt: dict, ev: dict) -> pd.Timestamp | None:
    # Try market-level then event-level
    candidates = [
        mkt.get("endDate"), mkt.get("endDateIso"), mkt.get("end_date"),
        mkt.get("closeTime"), mkt.get("closeTimeIso"),
        mkt.get("resolveTime"), mkt.get("resolveTimeIso"),
        ev.get("endDate"), ev.get("endDateIso"), ev.get("end_date"),
        ev.get("closeTime"), ev.get("closeTimeIso"),
    ]
    for c in candidates:
        dt = safe_dt(c)
        if dt is not None:
            return dt
    return None

def extract_markets_from_event(ev: dict) -> list[dict]:
    mkts = ev.get("markets")
    if isinstance(mkts, list):
        return mkts
    # Some payloads may embed markets differently; fallback
    return []

def inventory_crypto_markets(limit_events=200, max_events_pages=10, page_size=100):
    tag_id = get_crypto_tag_id()

    rows = []
    offset = 0
    pages = 0

    while pages < max_events_pages:
        events = gamma.list_events(tag_id=tag_id, active=True, closed=False, limit=page_size, offset=offset)
        if not events:
            break

        for ev in events:
            mkts = extract_markets_from_event(ev)
            for mkt in mkts:
                question = (mkt.get("question") or mkt.get("title") or mkt.get("name") or "").strip()
                if not question:
                    continue

                outcomes = normalize_outcomes(mkt.get("outcomes"))
                kind = classify_edge_or_range(question, outcomes)
                if kind not in {"edge", "range"}:
                    continue  # supported-only inventory for v1

                symbol = parse_underlying_symbol(question, coin_universe, project_universe)
                metric = parse_metric(question)
                res_dt = extract_resolution_date(mkt, ev)
                res_source, res_terms = extract_resolution_source_and_terms(mkt)
                routing = route_resolution_terms(res_source, res_terms)



                rows.append({
                    "market": question,
                    "kind": kind,  # edge or range
                    "symbol": symbol,
                    "metric":metric,
                    "resolution_date": res_dt,
                    "resolution_source": res_source,
                    "resolution_terms": res_terms,
                    "resolution_data_type": routing.data_type,
                    "resolution_interval": routing.interval,
                    "interval_source": routing.interval_source,
                    "routing_notes": routing.notes,
                })

                if len(rows) >= limit_events:
                    break
            if len(rows) >= limit_events:
                break

        if len(rows) >= limit_events:
            break

        offset += page_size
        pages += 1

    df = pd.DataFrame(rows)

    # Clean up
    if not df.empty:
        df["resolution_date"] = pd.to_datetime(df["resolution_date"], utc=True, errors="coerce")
        df = df.dropna(subset=["resolution_date"])
        df = df.sort_values(["kind", "resolution_date"]).reset_index(drop=True)

    return df

In [ ]:
df_inv = inventory_crypto_markets(limit_events=500)
df_inv.head(30)